In [0]:
# =============================================================
# 06_unity_catalog — Unity Catalog governance
# Author: oakville3456
# Branch: feature/priority5-unity-catalog
# Purpose: Register tables, add metadata, govern access
# =============================================================

# Check which catalogs are available to you
print("=== Available Catalogs ===")
spark.sql("SHOW CATALOGS").show(truncate=False)

# Check current catalog and database
print("=== Current Catalog ===")
spark.sql("SELECT current_catalog(), current_database()").show(truncate=False)

In [0]:
# Cell 2 — Explore catalog structure
print("=== Schemas (databases) in adb_retail_dev ===")
spark.sql("SHOW SCHEMAS IN adb_retail_dev").show(truncate=False)

# Check what tables already exist in default schema
print("=== Tables in adb_retail_dev.default ===")
spark.sql("SHOW TABLES IN adb_retail_dev.default").show(truncate=False)

In [0]:
# Cell 3 — Check existing tables in bronze, silver, gold schemas

for schema in ["bronze", "silver", "gold"]:
    print(f"=== Tables in adb_retail_dev.{schema} ===")
    spark.sql(f"SHOW TABLES IN adb_retail_dev.{schema}").show(truncate=False)

In [0]:
# Cell 4 — Query by table name instead of abfss:// paths

# Old way (what we've been doing)
# spark.read.format("delta").load("abfss://silver@saretailsalesdev.dfs.core.windows.net/sales")

# New way — Unity Catalog table name
print("=== Silver via Unity Catalog ===")
silver = spark.table("adb_retail_dev.silver.sales")
print(f"✅ Rows: {silver.count()}")
silver.printSchema()

print("\n=== Gold via Unity Catalog ===")
gold = spark.table("adb_retail_dev.gold.sales_daily")
print(f"✅ Rows: {gold.count()}")
gold.show(5)

In [0]:
# Cell 5 — Add column comments to Silver table
# This is how other team members know what each column means

spark.sql("""
    ALTER TABLE adb_retail_dev.silver.sales
    ALTER COLUMN order_id    COMMENT 'Unique order identifier — primary key'
""")
spark.sql("""
    ALTER TABLE adb_retail_dev.silver.sales
    ALTER COLUMN store_id    COMMENT 'Store identifier e.g. S01, S02'
""")
spark.sql("""
    ALTER TABLE adb_retail_dev.silver.sales
    ALTER COLUMN order_date  COMMENT 'Order date — NULL if source date was malformed'
""")
spark.sql("""
    ALTER TABLE adb_retail_dev.silver.sales
    ALTER COLUMN revenue     COMMENT 'Calculated as quantity * price'
""")
spark.sql("""
    ALTER TABLE adb_retail_dev.silver.sales
    ALTER COLUMN _ingested_at COMMENT 'Timestamp when row was ingested by Auto Loader'
""")
spark.sql("""
    ALTER TABLE adb_retail_dev.silver.sales
    ALTER COLUMN _source_file COMMENT 'Source CSV file path from ADLS raw-landing'
""")

# Verify comments were add

In [0]:
# Cell 6 — Verify column comments were added
spark.sql("DESCRIBE TABLE adb_retail_dev.silver.sales").show(truncate=False)

In [0]:
# Cell 7 — Add remaining column comments
spark.sql("""ALTER TABLE adb_retail_dev.silver.sales
    ALTER COLUMN product      COMMENT 'Product name e.g. Laptop, Monitor'""")
spark.sql("""ALTER TABLE adb_retail_dev.silver.sales
    ALTER COLUMN quantity     COMMENT 'Number of units ordered'""")
spark.sql("""ALTER TABLE adb_retail_dev.silver.sales
    ALTER COLUMN price        COMMENT 'Unit price in USD'""")
spark.sql("""ALTER TABLE adb_retail_dev.silver.sales
    ALTER COLUMN customer_id  COMMENT 'Customer identifier e.g. C001'""")
spark.sql("""ALTER TABLE adb_retail_dev.silver.sales
    ALTER COLUMN _rescued_data COMMENT 'Malformed fields captured by Auto Loader schema rescue'""")

# Verify all columns now documented
print("=== Silver — fully documented ===")
spark.sql("DESCRIBE TABLE adb_retail_dev.silver.sales").show(truncate=False)

In [0]:
# Cell 8 — Add table-level comment and tags

# Table comment
spark.sql("""
    COMMENT ON TABLE adb_retail_dev.silver.sales IS
    'Cleaned and deduplicated retail sales data.
     Source: Bronze Auto Loader ingestion from ADLS raw-landing.
     Deduplication key: order_id (latest _ingested_at wins).
     Bad dates set to NULL via try_to_date.
     Updated via Delta MERGE upsert pattern.'
""")

# Add tags (searchable labels in Unity Catalog UI)
spark.sql("""
    ALTER TABLE adb_retail_dev.silver.sales
    SET TAGS ('layer' = 'silver', 'domain' = 'retail', 'pii' = 'false')
""")

# Verify
print("=== Table details ===")
spark.sql("DESCRIBE TABLE EXTENDED adb_retail_dev.silver.sales") \
     .filter("col_name IN ('Comment', 'Tags')") \
     .show(truncate=False)

In [0]:
# Cell 9 — Verify tags + view full table properties

spark.sql("DESCRIBE TABLE EXTENDED adb_retail_dev.silver.sales") \
     .show(100, truncate=False)

In [0]:
# Cell 10 — Query across all three layers in one SQL statement
# This proves full Bronze → Silver → Gold lineage via Unity Catalog

spark.sql("""
    SELECT
        g.store_id,
        g.order_date,
        g.total_revenue,
        g.order_count,
        COUNT(s.order_id)    AS silver_orders,
        COUNT(b.order_id)    AS bronze_orders
    FROM adb_retail_dev.gold.sales_daily   g
    LEFT JOIN adb_retail_dev.silver.sales  s
        ON  s.store_id  = g.store_id
        AND s.order_date = g.order_date
    LEFT JOIN adb_retail_dev.bronze.sales  b
        ON  b.store_id  = g.store_id
    GROUP BY g.store_id, g.order_date, g.total_revenue, g.order_count
    ORDER BY g.order_date, g.store_id
""").show(10, truncate=False)